# Agente Predfy + H2O AutoML — MVP em notebook

**HyperCopa DISEC 2026 — Banco do Brasil**

## Pergunta de negócio

> **"Estamos estimando corretamente o prazo de uma EAP?"**
>
> Operacionalizada como classificação binária: **"esta EAP vai atrasar?"** (`Sim` / `Não`).

Onde **EAP = evento de contratação / Licitação Eletrônica** — cada linha do extrato é uma contratação processada pela DISEC sob a Lei 14.133/21.

## O que este notebook faz

Reproduz, em Jupyter, o último fluxo do app `app_agente_bb.py` (que roda em Streamlit) para que possa ser executado, lido e auditado de ponta a ponta sem depender da UI:

1. **Etapa 1 — Agente Predfy.** Um LLM (OpenAI function calling) inspeciona os CSVs que a área entregou, propõe a preparação e devolve **um único CSV pronto** (+ `meta.json` com `pergunta`, `target` e `task`).
2. **Etapa 2 — H2O AutoML.** O CSV final entra direto num cluster H2O local; o AutoML treina vários algoritmos, escolhe o melhor (líder do leaderboard) e calcula métricas no conjunto de teste.
3. **Etapa 3 — Persistência.** O modelo líder é salvo em disco junto com `metadata.json`, schema das features e amostra do conjunto teste.
4. **Etapa 4 — Consumo (MVP).** Carregamos o modelo salvo e fazemos `predict()` numa EAP nova, com **semáforo de risco** — exatamente como o Demo 2 do Streamlit.

## Dataset de demo já incluso 🎁

Para você rodar o notebook **sem trazer dados próprios**, deixei pronto:

```
dados_treino/demo_eaps_vai_atrasar.csv        (1.500 EAPs sintéticas, 15 colunas)
dados_treino/demo_eaps_vai_atrasar.meta.json  ({pergunta, target=vai_atrasar, task=classification})
```

A célula **"4. Carregue um (ou mais) CSV(s)"** já aponta para esse arquivo. Para rodar com extratos reais (DICOI, DISUP, DITEC, GECOI), basta trocar o caminho.

> **Atalho:** se você só quer ver o MVP rodando agora, pule direto para a **Etapa 2 — H2O AutoML** (Seção 7). O CSV de demo já está no formato final, então o agente é opcional para a primeira execução.

## Dois caminhos para conversar com o agente

| Caminho | Como funciona | Pré-requisito |
|---------|--------------|---------------|
| **A — OpenAI direto** | O notebook abre conexão com a API da OpenAI e roda o loop `chat ↔ tools` automaticamente. | `OPENAI_API_KEY` no `.env` da raiz |
| **B — Copilot do Teams (paste)** | Você conversa com o agente **Agente Predfy** dentro do Microsoft Copilot do Teams (sem chave OpenAI nossa); ele devolve um bloco de texto padronizado; cole esse bloco numa célula deste notebook e ele faz o resto. | Acesso ao Copilot Studio + agente publicado no Teams |

> Os dois caminhos terminam no **mesmo** ponto: um CSV em `dados_treino/` + `meta.json`. A partir daí, o restante do notebook é idêntico.

## 1. Instalação

Rode a célula abaixo apenas na primeira vez. Ela instala tudo o que `app_agente_bb.py` precisa, na mesma versão pinada em `requirements_app.txt`.

In [ ]:
%pip install -q \
    "pandas>=2.0" \
    "numpy" \
    "h2o>=3.46" \
    "scikit-learn>=1.3" \
    "openai>=1.50" \
    "python-dotenv>=1.0" \
    "plotly>=5.18" \
    "psutil"

# H2O exige Java 8/11/17 disponível no PATH. Verifique:
import shutil, subprocess
java = shutil.which("java")
print("java:", java)
if java:
    subprocess.run([java, "-version"])

## 2. Configuração — chave OpenAI (Caminho A) **OU** Copilot Teams (Caminho B)

Crie (se ainda não existir) um arquivo `.env` na raiz do projeto com **uma** linha:

```text
OPENAI_API_KEY=sk-proj-...
```

**Sem chave?** Sem problema — pule direto para a célula **"Caminho B — Copilot do Teams"** mais abaixo. O resto do notebook funciona igual.

> ⚠️ **Política DISEC:** se você não tem uma chave OpenAI corporativa liberada, **use o Caminho B**. O agente publicado no Copilot Studio do BB já roda dentro do tenant Microsoft, sem expor dados em APIs externas.

In [ ]:
from pathlib import Path
import os, json
from dotenv import load_dotenv

ROOT = Path.cwd()           # raiz do projeto (onde está o notebook)
PROMPT_PATH = ROOT / "docs" / "agente" / "system_prompt.md"
TOOLS_PATH  = ROOT / "docs" / "agente" / "tools_schema.json"
OUTPUT_DIR  = ROOT / "dados_treino"
MODELS_DIR  = ROOT / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

load_dotenv(ROOT / ".env")
tem_chave = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY definida?", tem_chave)
print("system_prompt.md existe?", PROMPT_PATH.exists())
print("tools_schema.json existe?", TOOLS_PATH.exists())

## 3. As 4 *tools* do agente

O agente da OpenAI não roda código — ele apenas **decide** o que chamar. Quem executa é este notebook (sandbox local). O contrato entre os dois é o `tools_schema.json`:

| Tool | Para quê serve |
|------|----------------|
| `ler_schema(nome_arquivo)` | Devolve colunas + dtypes de um CSV que está em memória. |
| `ler_amostra(nome_arquivo, n=5)` | Devolve as primeiras `n` linhas, em string. |
| `executar_pandas(codigo)` | Roda código pandas/numpy num **sandbox restrito** (sem `import`, sem rede, sem disco). O DataFrame final tem que ser atribuído à variável `resultado`. |
| `salvar_csv_final(df_id, nome, pergunta, target, task)` | Persiste o DF aprovado em `dados_treino/<nome>.csv` + sidecar `.meta.json`. **É o sinal verde** para a Etapa 2. |

As 3 primeiras são *iterativas* — o agente alterna entre elas dezenas de vezes até entender os dados. A 4ª é o ponto de saída: deve ser chamada **exatamente uma vez por conversa**.

In [ ]:
# Estado em memória — equivalente ao st.session_state do app Streamlit
import contextlib, io, uuid
import pandas as pd
import numpy as np

uploaded_dfs: dict[str, pd.DataFrame] = {}   # CSVs que o usuário trouxe
agent_dfs:    dict[str, pd.DataFrame] = {}   # DFs criados pelo agente via executar_pandas
final_csv_path: Path | None = None
final_meta: dict = {}

def _find_df(nome: str):
    return uploaded_dfs.get(nome) or agent_dfs.get(nome)

def tool_ler_schema(nome_arquivo: str) -> dict:
    df = _find_df(nome_arquivo)
    if df is None:
        return {"erro": f"arquivo '{nome_arquivo}' nao encontrado"}
    return {
        "nome": nome_arquivo,
        "linhas": int(len(df)),
        "colunas": {c: str(df[c].dtype) for c in df.columns},
    }

def _safe_str(val):
    if isinstance(val, bytes):
        try: return val.decode("utf-8")
        except UnicodeDecodeError: return val.decode("latin-1", errors="replace")
    if pd.isna(val): return ""
    return str(val)

def tool_ler_amostra(nome_arquivo: str, n: int = 5) -> dict:
    df = _find_df(nome_arquivo)
    if df is None:
        return {"erro": f"arquivo '{nome_arquivo}' nao encontrado"}
    n = max(1, min(int(n), 20))
    try:    amostra = df.head(n).map(_safe_str)
    except AttributeError: amostra = df.head(n).applymap(_safe_str)  # pandas <2.1
    return {"nome": nome_arquivo, "amostra": amostra.to_dict(orient="records")}

def tool_executar_pandas(codigo: str) -> dict:
    """Sandbox: exec() com builtins enxutos + pd/np apenas. Sem `import`."""
    dfs = {**uploaded_dfs, **agent_dfs}
    safe_builtins = {
        "len": len, "range": range, "list": list, "dict": dict, "set": set,
        "tuple": tuple, "str": str, "int": int, "float": float, "bool": bool,
        "print": print, "min": min, "max": max, "sum": sum, "abs": abs,
        "round": round, "sorted": sorted, "enumerate": enumerate, "zip": zip,
        "any": any, "all": all, "isinstance": isinstance, "type": type,
    }
    safe_globals = {"__builtins__": safe_builtins, "pd": pd, "np": np, "dfs": dfs}
    local_vars: dict = {}
    stdout_buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buf):
            exec(codigo, safe_globals, local_vars)
    except Exception as e:
        return {"ok": False, "erro": f"{type(e).__name__}: {e}",
                "stdout": stdout_buf.getvalue()[:2000]}
    resultado = local_vars.get("resultado")
    saida: dict = {"ok": True, "stdout": stdout_buf.getvalue()[:2000]}
    if isinstance(resultado, pd.DataFrame):
        df_id = f"df_{uuid.uuid4().hex[:8]}"
        agent_dfs[df_id] = resultado
        saida.update({"df_id": df_id, "shape": list(resultado.shape),
                      "colunas": list(resultado.columns)})
    elif isinstance(resultado, pd.Series):
        saida["resultado_serie"] = resultado.head(50).astype(str).to_dict()
    elif resultado is not None:
        saida["resultado_repr"] = str(resultado)[:2000]
    return saida

def tool_salvar_csv_final(df_id, nome_sugerido, pergunta="", target="", task="") -> dict:
    global final_csv_path, final_meta
    df = agent_dfs.get(df_id)
    if df is None:
        return {"erro": f"df_id '{df_id}' nao encontrado"}
    nome = nome_sugerido.strip().replace(" ", "_").replace("/", "_").replace("\\", "_")
    if not nome.endswith(".csv"):
        nome += ".csv"
    path = OUTPUT_DIR / nome
    df.to_csv(path, index=False)
    meta = {"pergunta": pergunta, "target": target, "task": task}
    path.with_suffix(".meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    final_csv_path = path
    final_meta = meta
    return {"arquivo": nome, "linhas": int(len(df)), "colunas": int(len(df.columns))}

TOOL_HANDLERS = {
    "ler_schema":      tool_ler_schema,
    "ler_amostra":     tool_ler_amostra,
    "executar_pandas": tool_executar_pandas,
    "salvar_csv_final": tool_salvar_csv_final,
}
print("tools registradas:", list(TOOL_HANDLERS))

## 4. Carregue um (ou mais) CSV(s) que a área entregou

Em produção isso vem do uploader do Streamlit. Aqui você só aponta um caminho. O loader é tolerante a separador `,`/`;` e encoding `utf-8/latin-1/cp1252` — exatamente como o app.

A célula abaixo já carrega o **CSV de demo** (`dados_treino/demo_eaps_vai_atrasar.csv` — 1.500 EAPs com 15 features). Para usar dados reais, troque o caminho na chamada `upload_csv(...)`.

In [ ]:
def upload_csv(caminho: str | Path):
    """Carrega 1 CSV pra `uploaded_dfs` detectando separador e encoding."""
    p = Path(caminho)
    head = p.read_bytes()[:4096]
    sep = ";" if head.count(b";") > head.count(b",") else ","
    df = None
    last = None
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, low_memory=False)
            break
        except (UnicodeDecodeError, pd.errors.ParserError) as e:
            last = e
    if df is None:
        raise RuntimeError(f"falhou: {last}")
    uploaded_dfs[p.name] = df
    print(f"✓ {p.name}: {len(df):,} linhas × {len(df.columns)} colunas")
    return df

# === Carga do CSV de demonstração (1.500 EAPs sintéticas) ===
DEMO_CSV = OUTPUT_DIR / "demo_eaps_vai_atrasar.csv"
if DEMO_CSV.exists():
    df_demo = upload_csv(DEMO_CSV)
    print("\nPrimeiras 5 linhas:")
    display(df_demo.head())
    print("\nDistribuição do target (vai_atrasar):")
    print(df_demo["vai_atrasar"].value_counts(normalize=True).round(3).to_string())
else:
    print("Demo CSV não encontrado. Rode primeiro: python gerar_demo_eaps.py")
    print("Ou use upload_csv('caminho/para/seu_extrato.csv')")

## 5 — Caminho A · Loop OpenAI ↔ tools (precisa de `OPENAI_API_KEY`)

Esta é a estrela do agente: a cada turno do usuário, o LLM decide se quer **pensar** (responder em texto) ou **agir** (chamar uma das 4 tools). Cada `tool_call` é executado **localmente** neste notebook, e a resposta volta pro LLM no mesmo histórico — até ele decidir parar de chamar tools e mandar a resposta final.

O `system_prompt.md` na pasta `docs/agente/` guia o comportamento (linguagem PT-BR, terminologia BB de Licitação Eletrônica/EAP, regras de não inventar coluna, etc.).

In [ ]:
# Use só se você tiver OPENAI_API_KEY definida.
MODELO_LLM = "gpt-5.2"   # "gpt-5.2-pro" para raciocínio mais profundo · "gpt-5.2-mini" econômico
MAX_TOOL_ITERS = 8        # iterações tool-calling por turno

def carregar_prompt_e_tools():
    sys_prompt = PROMPT_PATH.read_text(encoding="utf-8")
    tools = json.loads(TOOLS_PATH.read_text(encoding="utf-8"))["tools"]
    if uploaded_dfs:
        lista = "\n".join(f"- `{n}` ({len(d)} linhas × {len(d.columns)} cols)"
                          for n, d in uploaded_dfs.items())
        sys_prompt += f"\n\n## Arquivos disponiveis agora\n{lista}"
    return sys_prompt, tools

messages_history: list[dict] = []   # mantido entre chamadas a `falar_com_agente`

def falar_com_agente(pergunta: str) -> str:
    """Adiciona uma fala do usuário, roda o loop tool-calling e devolve a resposta final."""
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY não definida — use Caminho B mais abaixo.")
    from openai import OpenAI

    sys_prompt, tools = carregar_prompt_e_tools()
    messages_history.append({"role": "user", "content": pergunta})

    msgs_api = [{"role": "system", "content": sys_prompt}]
    for m in messages_history:
        msgs_api.append({k: v for k, v in m.items()
                         if k in ("role", "content", "tool_calls", "tool_call_id", "name")})

    client = OpenAI()
    final_text = ""

    for _ in range(MAX_TOOL_ITERS):
        resp = client.chat.completions.create(
            model=MODELO_LLM,
            messages=msgs_api,
            tools=tools,
            tool_choice="auto",
        )
        choice = resp.choices[0].message
        content = choice.content or ""
        tool_calls = choice.tool_calls or []

        assistant_msg = {"role": "assistant", "content": content}
        if tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in tool_calls
            ]
        msgs_api.append(assistant_msg)
        messages_history.append(assistant_msg)

        if not tool_calls:
            final_text = content
            break

        # Roda cada tool_call localmente
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            handler = TOOL_HANDLERS.get(tc.function.name)
            res = handler(**args) if handler else {"erro": f"tool '{tc.function.name}' nao existe"}
            tool_msg = {
                "role": "tool",
                "tool_call_id": tc.id,
                "name": tc.function.name,
                "content": json.dumps(res, ensure_ascii=False, default=str)[:6000],
            }
            msgs_api.append(tool_msg)
            messages_history.append(tool_msg)
            print(f"  🔧 {tc.function.name}({list(args)}) → {str(res)[:120]}")
    return final_text

# === Exemplo de uso com a demo ===
# Descomente para rodar o agente de fato (precisa OPENAI_API_KEY válida).
# resp = falar_com_agente(
#     "Recebi este extrato de Licitação Eletrônica. "
#     "Estamos estimando corretamente os prazos? "
#     "Quero prever quais EAPs vão atrasar."
# )
# print(resp)

## 5 — Caminho B · Cole o bloco do Copilot do Teams

**Quando usar:** se você não tem chave OpenAI corporativa, ou se a política da DISEC exige que o LLM rode dentro do tenant Microsoft.

**Como funciona:**
1. Abra o agente **Agente Predfy** dentro do Microsoft Copilot do Teams.
2. Anexe o CSV (ou cole uma amostra) e descreva sua pergunta preditiva.
3. O Copilot devolve um bloco padronizado com 7 campos.
4. Cole o bloco na variável `BLOCO_COPILOT` da célula abaixo, depois execute.

Formato esperado (palavras-chave em CAIXA-ALTA, dois-pontos, valor):

```text
PERGUNTA: Quais contratos vão atrasar?
TARGET: teve_atraso
TASK: classification
FEATURES_MANTER: porte_fornecedor,valor_contratado,num_aditivos,...
FILTRO: só contratos com vigência encerrada
JOINS: nenhum
TRATAMENTO_NULOS: numericos -> 0, texto -> 'desconhecido'
PASSO_A_PASSO_PANDAS: |
  contratos = dfs['contratos.csv']
  contratos = contratos[contratos['dt_vigencia_fim'] < '2025-01-01']
  resultado = contratos[['porte_fornecedor','valor_contratado','num_aditivos','teve_atraso']]
```

In [ ]:
import re

# Bloco já preenchido para a demo `demo_eaps_vai_atrasar.csv`.
# Substitua pelo bloco que o Copilot do Teams gerar para o seu CSV real.
BLOCO_COPILOT = """
PERGUNTA: Estamos estimando corretamente o prazo? Esta EAP vai atrasar?
TARGET: vai_atrasar
TASK: classification
FEATURES_MANTER: modalidade,tipo_servico,unidade_demandante,valor_estimado,num_etapas,num_participantes,tem_intercorrencia,urgencia,complexidade,status,dt_abertura_ano,dt_abertura_mes,etapas_duracao_media,etapas_interrompidas
FILTRO: nenhum
JOINS: nenhum
TRATAMENTO_NULOS: usar como esta (LightGBM/H2O lidam com NaN nativamente)
PASSO_A_PASSO_PANDAS: |
  eaps = dfs['demo_eaps_vai_atrasar.csv']
  cols = ['modalidade','tipo_servico','unidade_demandante','valor_estimado',
          'num_etapas','num_participantes','tem_intercorrencia','urgencia',
          'complexidade','status','dt_abertura_ano','dt_abertura_mes',
          'etapas_duracao_media','etapas_interrompidas','vai_atrasar']
  resultado = eaps[cols].copy()
"""

def processar_bloco_copilot(bloco: str, nome_csv_final: str = "treino_copilot"):
    """Equivalente ao botão 'Processar bloco e gerar CSV final' do app."""
    campos = {}
    codigo = ""
    modo_codigo = False
    for linha in bloco.splitlines():
        if modo_codigo:
            if linha.startswith((" ", "\t")):
                codigo += linha.lstrip() + "\n"
                continue
            modo_codigo = False
        m = re.match(r"^([A-Z_]+):\s*(.*)$", linha)
        if m:
            chave, valor = m.group(1), m.group(2).strip()
            if chave == "PASSO_A_PASSO_PANDAS" and valor in ("|", ""):
                modo_codigo = True
            else:
                campos[chave] = valor

    pergunta = campos.get("PERGUNTA", "").strip()
    target   = campos.get("TARGET", "").strip()
    task     = campos.get("TASK", "").strip().lower()

    if not pergunta or not target or task not in ("classification", "regression"):
        raise ValueError("Bloco inválido: PERGUNTA/TARGET/TASK obrigatórios; TASK ∈ {classification, regression}.")
    if not codigo.strip():
        raise ValueError("Bloco sem PASSO_A_PASSO_PANDAS.")
    if not uploaded_dfs:
        raise ValueError("Carregue um CSV com upload_csv() antes.")

    res = tool_executar_pandas(codigo)
    if not res.get("ok"):
        raise RuntimeError(f"Erro: {res.get('erro')}\n{res.get('stdout','')}")
    df_id = res["df_id"]
    df_resultante = agent_dfs[df_id]
    if target not in df_resultante.columns:
        raise ValueError(f"O código não produziu a coluna '{target}'. Colunas: {list(df_resultante.columns)}")
    return tool_salvar_csv_final(df_id=df_id, nome_sugerido=nome_csv_final,
                                 pergunta=pergunta, target=target, task=task)

# Para rodar a demo via Caminho B (sem chave OpenAI), descomente:
# saida = processar_bloco_copilot(BLOCO_COPILOT, nome_csv_final="treino_eaps_demo")
# print(saida)

## 6. Confira o que o agente entregou

Independente do caminho, o resultado mora em `dados_treino/<nome>.csv` + `dados_treino/<nome>.meta.json`. A célula abaixo é o ponto de **handoff** entre Etapa 1 (agente) e Etapa 2 (H2O).

In [ ]:
if final_csv_path is None:
    # Permite reabrir um treino antigo (caso já tenha rodado o agente em outra sessão)
    candidatos = sorted(OUTPUT_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidatos:
        raise SystemExit("Sem CSV em dados_treino/. Volte ao Caminho A ou B.")
    final_csv_path = candidatos[0]
    meta_p = final_csv_path.with_suffix(".meta.json")
    final_meta = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.exists() else {}

df_final = pd.read_csv(final_csv_path)
print(f"Arquivo: {final_csv_path.name}")
print(f"Pergunta: {final_meta.get('pergunta','—')}")
print(f"Target:   {final_meta.get('target','—')}")
print(f"Task:     {final_meta.get('task','—')}")
print(f"Shape:    {df_final.shape}")
df_final.head()

## 7. Etapa 2 — H2O AutoML

Aqui replicamos o que o app faz quando você clica **"🚀 Treinar modelo H2O"**:

- **Heap dinâmica** (50 % da RAM da máquina, mín 2 GB, máx 8 GB) — evita estourar memória em estações modestas.
- **Cluster local** subido com `h2o.init()` (a primeira vez baixa o backend Java; depois só reutiliza).
- **Split 80/20 estratificado** por `random_state=42` (mesmas linhas em toda execução).
- **Conversão para `H2OFrame`**; se for *classification*, força o target em `asfactor()` (categórico).
- **`H2OAutoML(max_runtime_secs=...)`** — exclui `StackedEnsemble` e `DeepLearning` no MVP por questão de latência/explicabilidade.
- **Métrica de ordenação:** `AUC` para classificação, `RMSE` para regressão.
- Saídas: leaderboard top-10, métricas no teste, importância das variáveis.

In [ ]:
import h2o
from h2o.automl import H2OAutoML

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    HEAP_GB = max(2, min(8, int(ram_gb * 0.5)))
except ImportError:
    HEAP_GB = 4
print(f"Heap H2O: {HEAP_GB} GB")

if not h2o.cluster() or not h2o.cluster().is_running():
    h2o.init(max_mem_size=f"{HEAP_GB}G", nthreads=-1)
else:
    print("Reutilizando cluster H2O ativo.")

In [ ]:
from datetime import datetime

BUDGET_SECS = 120          # orçamento de treino — para >100k linhas, ≥ 180s
target = final_meta["target"]
task   = final_meta.get("task", "classification")
assert target in df_final.columns, f"target '{target}' não está no CSV"

if len(df_final) > 500_000:
    df_final = df_final.sample(n=200_000, random_state=42)
    print(f"Amostragem: 200.000 linhas")

df_train = df_final.sample(frac=0.8, random_state=42)
df_test  = df_final.drop(df_train.index)
print(f"Split: {len(df_train):,} treino · {len(df_test):,} teste")

hf_train = h2o.H2OFrame(df_train)
hf_test  = h2o.H2OFrame(df_test)

if task == "classification":
    hf_train[target] = hf_train[target].asfactor()
    hf_test[target]  = hf_test[target].asfactor()

features = [c for c in df_final.columns if c != target]
print(f"Treinando AutoML — {len(features)} features, target='{target}', budget={BUDGET_SECS}s")

aml = H2OAutoML(
    max_runtime_secs=BUDGET_SECS,
    seed=42,
    sort_metric="AUC" if task == "classification" else "RMSE",
    exclude_algos=["StackedEnsemble", "DeepLearning"],
)
aml.train(x=features, y=target, training_frame=hf_train)
print("\n✅ Treino finalizado")

In [ ]:
lb = aml.leaderboard.as_data_frame(use_multi_thread=True)
leader = aml.leader
print(f"🏆 Vencedor: {leader.model_id}\n")
lb.head(10)

In [ ]:
perf_test = leader.model_performance(hf_test)
metrics = {}
if task == "classification":
    metrics["AUC"]      = float(perf_test.auc())
    metrics["LogLoss"]  = float(perf_test.logloss())
    try:    metrics["Accuracy"] = float(perf_test.accuracy()[0][1])
    except Exception: pass
else:
    metrics["RMSE"] = float(perf_test.rmse())
    metrics["MAE"]  = float(perf_test.mae())
    metrics["R2"]   = float(perf_test.r2())
print("Métricas no conjunto de teste:")
for k, v in metrics.items():
    print(f"  {k:8s} = {v:.4f}")

In [ ]:
varimp_df = None
try:
    varimp = leader.varimp(use_pandas=True)
    if varimp is not None:
        varimp_df = varimp.head(15)
        display(varimp_df)
except Exception as e:
    print("Sem varimp para este algoritmo:", e)

## 8. Persistência — modelo + metadata + amostra de teste

Esta célula reproduz o passo do app que **grava o modelo em `models/<slug>/`** com:

- o binário H2O (`leader`) salvo via `h2o.save_model(force=True)`,
- `metadata.json` com tudo que a Demo 2 precisa (target, task, métricas, schema das features),
- `test_sample.csv` com 5 linhas reais do conjunto **teste** — usadas no formulário "pré-preencher com exemplo real".

In [ ]:
import re as _re
def slugify(s: str) -> str:
    s = _re.sub(r"[^\w]+", "_", str(s).lower()).strip("_")
    return s or "modelo"

slug = slugify(target)
model_dir = MODELS_DIR / slug
model_dir.mkdir(parents=True, exist_ok=True)
for old in model_dir.glob("*"):                           # limpa modelos antigos do mesmo target
    if old.is_file() and old.name != "metadata.json":
        try: old.unlink()
        except Exception: pass

saved_path = h2o.save_model(model=leader, path=str(model_dir), force=True)
saved_filename = Path(saved_path).name

feature_info: dict = {}
for col in features:
    s = df_train[col]
    if pd.api.types.is_numeric_dtype(s):
        feature_info[col] = {"type": "number",
            "min":    float(s.min()) if pd.notna(s.min()) else 0.0,
            "max":    float(s.max()) if pd.notna(s.max()) else 0.0,
            "mean":   float(s.mean()) if pd.notna(s.mean()) else 0.0,
            "median": float(s.median()) if pd.notna(s.median()) else 0.0}
    else:
        n_unique = s.nunique(dropna=True)
        if n_unique <= 30:
            feature_info[col] = {"type": "select",
                "values": sorted(s.dropna().astype(str).unique().tolist())}
        else:
            feature_info[col] = {"type": "text"}

modelo_metadata = {
    "slug": slug,
    "target": target,
    "task": task,
    "pergunta": final_meta.get("pergunta", ""),
    "leader_id": str(leader.model_id),
    "saved_filename": saved_filename,
    "metrics": metrics,
    "treinado_em": datetime.now().isoformat(timespec="seconds"),
    "n_train": len(df_train),
    "n_test":  len(df_test),
    "features": features,
    "feature_info": feature_info,
}
(model_dir / "metadata.json").write_text(
    json.dumps(modelo_metadata, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")
df_test.sample(n=min(5, len(df_test)), random_state=42)\
       .to_csv(model_dir / "test_sample.csv", index=False, encoding="utf-8")

print(f"✓ Modelo salvo em: {model_dir}")
print(f"  • {saved_filename}")
print(f"  • metadata.json")
print(f"  • test_sample.csv")

## 9. Etapa 4 — Demo 2: consumir o modelo num caso novo (MVP)

Aqui o agente já saiu de cena. **Qualquer pessoa** que tenha o conteúdo da pasta `models/<slug>/` consegue rodar isto — não precisa nem da chave OpenAI nem do CSV de treino.

Pipeline:

1. Lê `metadata.json` → sabe o target, a task e o schema das features.
2. Sobe um cluster H2O (4 GB) e dá `h2o.load_model(...)`.
3. Monta um `pd.DataFrame([1 linha])` com a entrada do usuário.
4. Converte → `H2OFrame` → `predict()`.
5. Para classificação, pinta um **semáforo de risco** (verde <10 %, amarelo <25 %, vermelho ≥25 %).
6. Para regressão, mostra valor previsto ± RMSE.

In [ ]:
# Carrega metadata + modelo do disco (poderia ser num notebook completamente diferente)
mdl = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
if not h2o.cluster() or not h2o.cluster().is_running():
    h2o.init(max_mem_size="4G", nthreads=-1)
leader_carregado = h2o.load_model(str(model_dir / mdl["saved_filename"]))
print(f"Modelo carregado: {mdl['leader_id']}")
print(f"Pergunta: {mdl['pergunta']}")
print(f"Target:   {mdl['target']} ({mdl['task']})")
print(f"Métricas treino: {mdl['metrics']}")

In [ ]:
def prever(valores: dict) -> dict:
    """
    `valores` é um dict {feature_name: valor} cobrindo TODAS as features de mdl['features'].
    Retorna dict com classe/probabilidade (ou valor numérico) e semáforo de risco.
    """
    df_in = pd.DataFrame([valores])
    hf    = h2o.H2OFrame(df_in)
    pred  = leader_carregado.predict(hf).as_data_frame(use_multi_thread=True)

    if mdl["task"] == "classification":
        classe = str(pred["predict"].iloc[0])
        prob_cols = [c for c in pred.columns if c != "predict"]
        prob = float(pred[prob_cols[-1]].iloc[0]) if len(prob_cols) >= 2 else None
        if prob is None:
            sem, acao = "—", "Sem probabilidade."
        elif prob < 0.10:
            sem, acao = "🟢 Verde",     "Risco baixo — fluxo padrão."
        elif prob < 0.25:
            sem, acao = "🟡 Amarelo",   "Risco moderado — checkpoint extra."
        else:
            sem, acao = "🔴 Vermelho",  "Risco alto — revisar TR e governança."
        return {"tipo": "classification", "classe": classe,
                "probabilidade": prob, "semaforo": sem, "acao": acao}
    else:
        valor = float(pred["predict"].iloc[0])
        rmse  = mdl["metrics"].get("RMSE")
        return {"tipo": "regression", "valor": valor, "erro_tipico": rmse}

# Exemplo: sorteia um caso real do conjunto teste (que o modelo NUNCA viu) e prevê
test_csv = model_dir / "test_sample.csv"
if test_csv.exists():
    exemplo = pd.read_csv(test_csv).sample(1, random_state=0).iloc[0].to_dict()
    gabarito = exemplo.pop(mdl["target"], None)
    valores = {f: exemplo[f] for f in mdl["features"] if f in exemplo}
    resultado = prever(valores)
    print("Entrada:")
    for k, v in valores.items():
        print(f"  {k}: {v}")
    print("\nResultado:", resultado)
    print(f"Gabarito real: {gabarito}")
else:
    print("Sem test_sample.csv — chame prever({...}) com valores manuais.")

## 10. Encerrar o cluster (opcional)

O cluster H2O fica vivo enquanto o kernel do Jupyter estiver de pé. Para liberar memória entre experimentos:

In [ ]:
# h2o.cluster().shutdown(prompt=False)

## 11. Próximos passos

- **Reentrenar a cada novo trimestre fiscal** (ou após mudança de política de Licitação Eletrônica).
- Se o `AUC < 0.70` ou `R² < 0.60`, volte ao agente e peça **outro extrato** à área — provavelmente faltam features.
- Para **explicabilidade** num caso individual, troque `leader.predict(hf)` por `leader.predict_contributions(hf)` (SHAP nativo do H2O).
- Em produção, este notebook vira **2 endpoints FastAPI** (já estão em `app_agente_bb.py`):
  - `POST /preparar` (Etapa 1, agente) → devolve handle do CSV.
  - `POST /prever`  (Etapa 4, MVP)    → devolve classe + semáforo, dado um JSON de entrada.
- O caminho B (Copilot Teams) é o caminho oficial DISEC para produção: rode o agente **dentro do tenant Microsoft** e use este notebook só na metade analítica.